# pyAVS quickstart: MEG + eye-tracking on the AVS dataset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KietzmannLab/pyavs/blob/main/examples/pyavs_colab_quickstart.ipynb)

A ~15 minute, hands-on first contact with the **AVS dataset** (MEG + eye-tracking + MRI, 5
participants viewing MS-COCO natural scenes) via the **pyAVS** Python package. Using one
subject/session, this notebook covers:

1. Loading raw MEG and eye-tracking data
2. Loading a scene image together with the fixations made on it
3. A sensor-space comparison: MEG responses to fixations on **dogs** vs **airplanes**

This is a quick "raw ingredients + one precomputed derivative" tour, not a rerun of the full
preprocessing pipeline (Maxwell filtering, ICA, epoching) — for that, and for a broader tour
of the package, see **Where to go next** at the end.

> **Scope note:** the dog-vs-airplane MEG contrast in Section 5 is computed for a single
> subject and is meant to illustrate the tooling, not as a statistical claim about object-category
> representation in MEG — no significance testing is performed here.

## 1. Setup

In [9]:
#%pip install -q git+https://github.com/KietzmannLab/pyavs.git

## 2. Get the data

This notebook uses **subject 1, session 1** only, from the public `avs-public` dataset on AWS
([Open Data Sponsorship Program](https://registry.opendata.aws/), bucket `kietzmannlab-avs`,
`us-west-2`). Data loads through pyAVS's built-in remote client,
[`pyavs.open_remote()`](https://github.com/KietzmannLab/pyavs/blob/main/pyavs/remote/client.py)
(`pyavs.remote.AVSRemote`): it fetches each file a call needs directly from the public bucket
on first use — no manual staging, no AWS account or credentials (the bucket is
public/anonymous-read) — and caches it locally (default `~/.cache/pyavs/kietzmannlab-avs`), so
repeat calls and reruns skip the network.

In [10]:
import pyavs

avs = pyavs.open_remote()
avs

AVSRemote(S3Store(bucket='kietzmannlab-avs', cache_root='/Users/atlas/.cache/pyavs/kietzmannlab-avs'))

## 3. Raw ingredients: MEG, eye events, and a scene image

Start with one MEG recording block and a quick look at the signal.

In [11]:
raw = avs.load_meg_raw(subject_id=1, session=1, run=1, preload=True, verbose=True)
raw

[2026-08-20 09:28:17] pyavs.remote.store - INFO - Cached: sub-01/ses-01/meg/as01a01.fif (296.44 MB) at /Users/atlas/.cache/pyavs/kietzmannlab-avs/sub-01/ses-01/meg/as01a01.fif
[2026-08-20 09:28:17] pyavs.dataloader.meg - INFO - Loading MEG data from: /Users/atlas/.cache/pyavs/kietzmannlab-avs/sub-01/ses-01/meg/as01a01.fif
Opening raw data file /Users/atlas/.cache/pyavs/kietzmannlab-avs/sub-01/ses-01/meg/as01a01.fif...
    Read a total of 6 projection items:
        grad_ssp_upright2.fif : PCA-v1 (1 x 306)  idle
        grad_ssp_upright2.fif : PCA-v2 (1 x 306)  idle
        grad_ssp_upright2.fif : PCA-v3 (1 x 306)  idle
        mag_ssp_upright2.fif : PCA-v1 (1 x 306)  idle
        mag_ssp_upright2.fif : PCA-v2 (1 x 306)  idle
        mag_ssp_upright2.fif : PCA-v3 (1 x 306)  idle
    Range : 21000 ... 254999 =     21.000 ...   254.999 secs
Ready.
Reading 0 ... 233999  =      0.000 ...   233.999 secs...


/Users/atlas/Documents/Documents_atlas/PhD/code/pyavs_conversion/pyavs/pyavs/dataloader/meg.py:58: RuntimeWarning: This filename (/Users/atlas/.cache/pyavs/kietzmannlab-avs/sub-01/ses-01/meg/as01a01.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  return mne.io.read_raw_fif(meg_path, preload=preload, verbose=verbose)


<Raw | as01a01.fif, 310 x 234000 (234.0 s), ~560.0 MiB, data loaded>

In [12]:
raw.compute_psd(fmax=100, picks="meg").plot();

Effective window size : 2.048 (s)
Plotting power spectral density (dB=True).


/Users/atlas/miniforge3/envs/avs/lib/python3.11/site-packages/mne/viz/utils.py:160: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  (fig or plt).show(**kwargs)


Now the eye-tracking side: cleaned fixation/saccade events for the same subject/session,
enriched with trial/block/scene context (which scene, which block, time within the trial,
and so on). Object labels for what was fixated (e.g. "dog") aren't part of this
general-purpose events table — Section 5 pulls those separately from the fixation-locked
epochs' own metadata.

`avs.load_experiment_log`/`avs.load_eye_events` fetch the underlying files from S3 on first
use; the enrichment join itself (`load_and_enrich_eye_events`) then runs through pyAVS's
regular local loader, pointed at the same cache via `avs.data_path`.

In [13]:
avs.load_experiment_log(subject_id=1, session=1)
avs.load_eye_events(subject_id=1, session=1)

explog_df, events_df = pyavs.load_and_enrich_eye_events(subjects=[1], sessions=[1], data_path=avs.data_path)

fixations = events_df[(events_df["type"] == "fixation") & (events_df["recording"] == "scene")]
print(f"{len(fixations)} scene-viewing fixations, on {fixations['sceneID'].nunique()} distinct scenes")
fixations.head()

[2026-08-20 09:28:20] pyavs.remote.store - INFO - Cached: sub-01/ses-01/beh/as_exp_data_1_1_3_0.parquet (54.42 KB) at /Users/atlas/.cache/pyavs/kietzmannlab-avs/sub-01/ses-01/beh/as_exp_data_1_1_3_0.parquet
[2026-08-20 09:28:22] pyavs.remote.store - INFO - Cached: derivatives/pyavs/sub-01/ses-01/eyetrack/as_s1_el_events.parquet (520.89 KB) at /Users/atlas/.cache/pyavs/kietzmannlab-avs/derivatives/pyavs/sub-01/ses-01/eyetrack/as_s1_el_events.parquet
[2026-08-20 09:28:23] pyavs.remote.store - INFO - Cached: derivatives/pyavs/sub-01/ses-01/eyetrack/as_s1_el_msgs.parquet (92.82 KB) at /Users/atlas/.cache/pyavs/kietzmannlab-avs/derivatives/pyavs/sub-01/ses-01/eyetrack/as_s1_el_msgs.parquet


Processing subjects:   0%|          | 0/1 [00:00<?, ?it/s]

[2026-08-20 09:28:23] pyavs.dataloader.eye - INFO - Subject 1, session 1: 11211 events, 300 trials
[2026-08-20 09:28:23] pyavs.dataloader.eye - INFO - Fixing multi-saccades: 6873 scene events before
[2026-08-20 09:28:24] pyavs.dataloader.eye - INFO - After fixing: 6257 scene events












Processing subjects: 100%|██████████| 1/1 [00:01<00:00,  1.38s/it]

[2026-08-20 09:28:24] pyavs.dataloader.eye - INFO - Total events loaded: 10595
2983 scene-viewing fixations, on 300 distinct scenes


,duration,type,end_time,peak_velocity,mean_gx,start_gy,start_gx,end_gx,rms,start_time,...,sceneID,time_in_trial,block,trial_per_block,caption_task,multi_saccade,fix_sequence,fix_sequence_from_last,sac_sequence,sac_sequence_from_last
1,0.222,fixation,93026.430,NaN,598.653822,NaN,NaN,NaN,0.427039,93026.208,...,11299.0,0.327,1.0,1.0,1,no,0.0,-11.0,NaN,NaN
3,0.272,fixation,93026.710,NaN,616.935900,NaN,NaN,NaN,0.508870,93026.438,...,11299.0,0.557,1.0,1.0,1,no,1.0,-10.0,NaN,NaN
5,0.508,fixation,93027.232,NaN,558.035753,NaN,NaN,NaN,0.806423,93026.724,...,11299.0,0.843,1.0,1.0,1,no,2.0,-9.0,NaN,NaN
7,0.269,fixation,93027.516,NaN,566.902601,NaN,NaN,NaN,0.534730,93027.247,...,11299.0,1.366,1.0,1.0,1,no,3.0,-8.0,NaN,NaN
9,0.239,fixation,93027.767,NaN,569.025424,NaN,NaN,NaN,0.601238,93027.528,...,11299.0,1.647,1.0,1.0,1,no,4.0,-7.0,NaN,NaN


## 4. Eye movements on the scene

`EyeTrackingPlotter` overlays a subject's fixations directly on the COCO scene image they
viewed. `plot_scene` silently skips scenes the subject never actually viewed, so `scene_id`
below is pulled from the loaded fixation data rather than hardcoded.

In [14]:
from pyavs.config.config import PyAVSConfig
from pyavs.visualization.events_on_scene import EyeTrackingPlotter

plotter = EyeTrackingPlotter(subjects=1, sessions=1, config=PyAVSConfig(), data_path=avs.data_path)
scene_id = int(plotter.df["sceneID"].iloc[0])

[2026-08-20 09:28:24] pyavs.visualization.events_on_scene - INFO - Loading eye tracking data for subjects [1], sessions [1]...


Processing subjects:   0%|          | 0/1 [00:00<?, ?it/s]

[2026-08-20 09:28:24] pyavs.dataloader.eye - INFO - Subject 1, session 1: 11211 events, 300 trials


[2026-08-20 09:28:25] pyavs.dataloader.eye - INFO - Fixing multi-saccades: 6873 scene events before
[2026-08-20 09:28:25] pyavs.dataloader.eye - INFO - After fixing: 6257 scene events












Processing subjects: 100%|██████████| 1/1 [00:01<00:00,  1.34s/it]

[2026-08-20 09:28:26] pyavs.dataloader.eye - INFO - Total events loaded: 10595
[2026-08-20 09:28:26] pyavs.visualization.events_on_scene - INFO - Loaded 2983 fixations from 300 scenes
[2026-08-20 09:28:26] pyavs.visualization.events_on_scene - INFO - Subjects: [1]
[2026-08-20 09:28:26] pyavs.visualization.events_on_scene - INFO - Sessions: [1]


**Note on image availability:** the public release doesn't ship per-image scene JPEGs at
all — COCO/Flickr photos carry no redistribution license, so there's no `stimuli/images/`
directory on S3 (or anywhere else) to wait for. Instead, pyAVS reconstructs each scene image
on demand straight from COCO's own hosting: it looks up the image's original `coco_url` in
the small `avs_scenes_all_licenses.parquet` lookup table, downloads it, and reapplies the same
center-crop/resize used to build the original MEG-size stimuli (`Layout.ensure_scene_image`,
used internally by `EyeTrackingPlotter.plot_scene` below). That lookup table is itself a real
release file and needs fetching from S3 once, the same way `avs.load_experiment_log`/
`avs.load_eye_events` fetch theirs above — the cell below does that before plotting.

In [15]:
avs.store.fetch(plotter.layout.scene_licenses().relative_to(avs.data_path).as_posix())

plotter.plot_scene(scene_id, subject=1)

RemoteFileNotFoundError: No object at https://kietzmannlab-avs.s3.us-west-2.amazonaws.com/stimuli/avs_scenes_all_licenses.parquet

## 5. MEG responses to dog vs. airplane fixations

`avs.load_epochs(...)` fetches the precomputed `fixation_scene` epochs from S3 and returns an
`mne.Epochs` with per-fixation metadata attached (including `object_label`) — but built with
placeholder channel names (`MEG0001`, `MEG0002`, …), since the shipped arrays carry no channel
identity of their own. Every AVS session uses the same fixed 306-channel Neuromag/MEGIN
system, though, so we rebuild a channel-accurate `mne.EpochsArray` by borrowing the real
layout and digitization from the raw block already loaded in Section 3, and reuse the fetched
arrays and metadata as-is.

In [ ]:
import numpy as np
import mne

remote_epochs = avs.load_epochs(subject_id=1, session=1, event_type="fixation_scene")

grad_picks = mne.pick_types(raw.info, meg="grad")
mag_picks = mne.pick_types(raw.info, meg="mag")
epochs_info = mne.pick_info(raw.info, np.concatenate([grad_picks, mag_picks]))
with epochs_info._unlock():
    epochs_info["sfreq"] = remote_epochs.info["sfreq"]  # h5's native rate (500 Hz); raw's own sfreq (1000 Hz) doesn't apply here

epochs_data = remote_epochs.get_data()  # (n_epochs, 306, n_times); grad channels then mag, matching epochs_info above

n_epochs = epochs_data.shape[0]
sfreq = epochs_info["sfreq"]
events = np.column_stack([
    np.arange(n_epochs) * int(sfreq),
    np.zeros(n_epochs, dtype=int),
    np.ones(n_epochs, dtype=int),
])
epochs = mne.EpochsArray(
    epochs_data, epochs_info, events=events, tmin=remote_epochs.tmin,
    event_id={"fixation": 1}, verbose=False,
)
epochs.metadata = remote_epochs.metadata

metadata = epochs.metadata
n_dog, n_airplane = (metadata["object_label"] == "dog").sum(), (metadata["object_label"] == "airplane").sum()
print(f"dog: {n_dog} fixations, airplane: {n_airplane} fixations (of {n_epochs} total)")

Time series (mean over magnetometers):

In [ ]:
dog_evoked = epochs['object_label == "dog"'].average()
airplane_evoked = epochs['object_label == "airplane"'].average()

mne.viz.plot_compare_evokeds(
    {"dog": dog_evoked, "airplane": airplane_evoked},
    picks="mag", combine="mean", title="Fixation-locked MEG response (n=1 subject, illustrative only)",
);

Sensor-space topography at a fixed latency after fixation onset:

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, evoked, label in zip(axes, [dog_evoked, airplane_evoked], ["dog", "airplane"]):
    evoked.plot_topomap(times=0.15, ch_type="mag", axes=ax, colorbar=False, show=False)
    ax.set_title(label)
fig.suptitle("Fixation-locked MEG topography, 150 ms post-fixation-onset (mag)")
plt.show()

## 6. Where to go next

This notebook only scratched the surface: raw data plus one precomputed derivative, no
preprocessing. For more:

- **Full MEG + eye-tracking preprocessing walkthrough** (Maxwell filtering, ICA, event
  alignment, epoching) via `AVSComposer`:
  [`docs/tutorials/meg_eye_workflow.rst`](https://github.com/KietzmannLab/pyavs/blob/main/docs/tutorials/meg_eye_workflow.rst)
- **More worked examples** (object detection, source reconstruction, population codes, config
  reproducibility): [`examples/`](https://github.com/KietzmannLab/pyavs/tree/main/examples)
- **Full analysis pipelines** used in the AVS paper (representational similarity analysis,
  ANN-to-MEG encoding, source-projected fixation ERFs, decoding):
  [`scripts/`](https://github.com/KietzmannLab/pyavs/tree/main/scripts)
- **Citing the dataset or pyAVS**:
  [`docs/reference/citation.rst`](https://github.com/KietzmannLab/pyavs/blob/main/docs/reference/citation.rst)

*(The full rendered documentation site is not live yet as of this writing — the links above
point at the source `.rst`/example files on GitHub in the meantime.)*